In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd # all the tasks in part one need pandas
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()#normal method in pandas

In [ ]:
# Task 3: Write your code here:
df.info()#normal method in pandas

In [ ]:
# Task 4: Write your code here:
df.describe()#normal method in pandas

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt #import plotlib for plotting
plt.figure(figsize=(10, 5))#size of figure
plt.plot(df['delivery_time'].dropna(), bins=30, edgecolor='black')#what to plot
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()



In [ ]:
# Task 1: Write your code here:
df_clean = df.drop(["Order_ID"] , axis = 1)#dropping ID

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_clean.isnull().sum()) #seeing how many null values we have
to_null = ['Weather' , 'Traffic_Level' , 'Time_of_Day'] #colls that will be none

df_clean[to_null] = df_clean[to_null].fillna('none')#making none

df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())#filling this collum with the mean

df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df['Delivery_Time'].mean())#filling this collum with the mean

print("Missing values:")
print(df_clean.isnull().sum()) #checking coloms

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_clean.duplicated().sum()#cheking how many duplikate

if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)#removing if there are any
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = le.fit_transform(df_clean[col]) #encoding each string elemnt

#df_clean = onehot_encoder.fit_transform()
df_clean.head()

In [ ]:
# Task 5: Write your code here:
import numpy as np
from sklearn.preprocessing import StandardScaler , OneHotEncoder #import scaler

scaler = StandardScaler()
features = df_clean.columns #getting cols

df_clean[features] = scaler.fit_transform(df_clean[features])#scaling cols
df_clean

In [ ]:
# Task 6: Write your code here:
#there is no imbalance since the target is a number (dilivery time)

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop(columns=["Delivery_Time"])#splitting the data
y = df_clean['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)# kfold no imbalance

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=100, max_depth=15,class_weight='balanced', random_state=42)# the model randomforset


y_pred = model.predict(X_train)

mae = mean_absolute_error(y_test, y_pred)#loss function

print(f"MAE:  ${mae:,.2f}")


In [ ]:
# Task 1: Write your code here:
importances = {}# to save most important feacures
sklearn_models = {"Random Forest": RandomForestClassifier(
      n_estimators=320,
      max_depth=4)}
importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()#plotting the feacures


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test.numpy(), y_pred.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('dilivery time', fontsize=12)
plt.ylabel('Predicted dilivery time', fontsize=12)
plt.title('Predicted vs Actual dilivery time', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()#plotting

In [ ]:
# Task Bonus: Write your code here:
from sklearn.tree import CatBoostRegressor#i don't have time
models = {
    'Decision Tree': CatBoostRegressor(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),#our two models
}
odel_losses = {}
num_classes = len(np.unique(y_encoded))
print("Starting K-Fold Cross-Validation for each model...")
model_losses = {}
# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = categorical_cross_entropy(y_val_one_hot, y_pred_proba)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)